# Sistema de Contagem de Pessoas Únicas com Re-ID (Fase Final)

Este notebook implementa um sistema de visão computacional capaz de identificar e contar pessoas únicas através de uma webcam. 
Diferente de sistemas de contagem simples, este utiliza **Re-Identificação (Re-ID)** para reconhecer a mesma pessoa mesmo após ela sair e voltar para o campo de visão.

### Tecnologias Utilizadas:
1. **YOLOv11**: Detecção de objetos em tempo real.
2. **MobileNetV3 (Feature Extractor)**: Gera uma "assinatura visual" única para cada pessoa detectada.
3. **Similaridade de Cosseno**: Compara novas detecções com o banco de dados de pessoas conhecidas.
4. **Filtro de Estabilidade**: Garante que IDs temporários não inflem o contador final.

In [ ]:
import cv2
import torch
import os
import time
import numpy as np
from ultralytics import YOLO
from torchvision import models, transforms
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display

# Configurações de Ambiente
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
device = "cuda" if torch.cuda.is_available() else "cpu"

def load_models():
    """Carrega e configura os modelos de Detecção (YOLO) e Extração de Características (MobileNet)"""
    print("--- Carregando Modelos ---")
    
    # YOLOv11 para detecção rápida
    yolo = YOLO("yolo11n.pt")
    yolo.to(device)
    
    # MobileNetV3 Small otimizado para extração de embeddings
    reid = models.mobilenet_v3_small(weights='DEFAULT')
    reid.classifier = torch.nn.Identity() # Remove camada de classe para obter o vetor de características puro
    reid.to(device)
    reid.eval()
    
    return yolo, reid

# Pipeline de pré-processamento para o modelo de Re-ID
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

yolo_model, reid_model = load_models()
print(f"✅ Setup concluído com sucesso no device: {device}")

### Loop de Processamento com Lógica de Re-ID Estável

A lógica abaixo gerencia um banco de dados dinâmico de assinaturas visuais. 
Para cada nova detecção, o sistema busca a melhor correspondência no banco. 
Se a similaridade for baixa, uma nova identidade é criada apenas após um período de estabilização.

In [ ]:
def start_counting():
    # --- Parâmetros de Calibração ---
    REID_THRESHOLD = 0.72       # Similaridade mínima para considerar a mesma pessoa
    MIN_STABILITY_FRAMES = 20   # Frames mínimos de visão constante para validar um ID no contador
    MAX_SAMPLES_PER_ID = 15     # Quantas variações visuais guardamos de cada pessoa
    
    # --- Estado do Sistema ---
    person_db = {}     # Dicionário: {id: {'signatures': [], 'frames_seen': 0}}
    next_global_id = 1 
    prev_time = 0
    
    # Interface Notebook
    image_widget = widgets.Image(format='jpeg', width=640, height=480)
    display(image_widget)
    
    cap = cv2.VideoCapture(0)
    print("🔄 Contador e Banco de Dados resetados. Iniciando...")
    
    try:
        while cap.isOpened():
            success, frame = cap.read()
            if not success: break

            # 1. Detecção YOLO (Filtro 'person')
            results = yolo_model(frame, verbose=False, classes=[0], conf=0.65)
            annotated_frame = frame.copy()
            
            if results and len(results[0].boxes) > 0:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                
                for box in boxes:
                    x1, y1, x2, y2 = map(int, box)
                    person_crop = frame[max(0, y1):y2, max(0, x1):x2]
                    if person_crop.size == 0: continue
                    
                    # 2. Extração de Assinatura Visual (Embedding)
                    input_tensor = preprocess(person_crop).unsqueeze(0).to(device)
                    with torch.no_grad():
                        embedding = reid_model(input_tensor).cpu().numpy().reshape(1, -1)
                    
                    # 3. Busca no Banco de Dados (Best Match)
                    matched_id = None
                    highest_similarity = -1
                    
                    for pid, data in person_db.items():
                        # Compara a imagem atual com todas as variações salvas dessa pessoa
                        for sig in data['signatures']:
                            sim = cosine_similarity(embedding, sig)[0][0]
                            if sim > REID_THRESHOLD and sim > highest_similarity:
                                highest_similarity = sim
                                matched_id = pid
                    
                    # 4. Atualização de Identidade
                    if matched_id is not None:
                        person_db[matched_id]['frames_seen'] += 1
                        # Atualiza o banco visual com novas poses para melhorar precisão futura
                        if highest_similarity < 0.90 and len(person_db[matched_id]['signatures']) < MAX_SAMPLES_PER_ID:
                            person_db[matched_id]['signatures'].append(embedding)
                    else:
                        # Nova pessoa em potencial encontrada
                        matched_id = next_global_id
                        next_global_id += 1
                        person_db[matched_id] = {'signatures': [embedding], 'frames_seen': 1}
                    
                    # 5. Visualização na Tela
                    is_confirmed = person_db[matched_id]['frames_seen'] >= MIN_STABILITY_FRAMES
                    color = (0, 255, 0) if is_confirmed else (0, 165, 255)
                    status_text = f"ID #{matched_id}" if is_confirmed else f"Analisando... ({person_db[matched_id]['frames_seen']}/{MIN_STABILITY_FRAMES})"
                    
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(annotated_frame, status_text, (x1, y1-10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # 6. Cálculo do Contador Único (Apenas estáveis)
            total_confirmed = len([p for p in person_db.values() if p['frames_seen'] >= MIN_STABILITY_FRAMES])
            
            # Interface de Status (FPS e Total)
            curr_time = time.time()
            fps = 1 / (curr_time - prev_time) if (curr_time - prev_time) > 0 else 0
            prev_time = curr_time

            cv2.rectangle(annotated_frame, (0, 0), (320, 100), (0, 0, 0), -1)
            cv2.putText(annotated_frame, f"FPS: {int(fps)}", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(annotated_frame, f"Total Unico: {total_confirmed}", (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            # Encode e Display
            _, buffer = cv2.imencode('.jpg', annotated_frame)
            image_widget.value = buffer.tobytes()
            
    except KeyboardInterrupt: pass
    finally:
        cap.release()
        print(f"--- Sessão Encerrada ---\nTotal de pessoas confirmadas: {total_confirmed}")

start_counting()